# Day 01: The Scalar

**Series:** PyTorch from Scratch: The Zero-to-One Framework  
**Project:** VukTorch

Today we build the smallest possible unit of a deep learning framework: a single scalar value that knows:

- its numeric data
- its gradient
- how it was produced
- how to send local derivatives backward


## Learning Goal

By the end of this notebook, the object model should feel obvious:

- a scalar is just a number with memory
- every operation creates a new node in a computation graph
- local derivatives are enough to recover full gradients
- backpropagation is just the chain rule applied repeatedly


## 1. What Is a Scalar?

A scalar is the simplest kind of tensor: just one number.

In a framework, we do not want to store only the number itself. We also want to store the gradient associated with that number.

So instead of working with plain Python floats, we will wrap them in a small class.

Use the next cell to show the limitation of ordinary floats first, then introduce a `Value` object with `data` and `grad`.


## 2. A Tiny Computation Graph

Suppose we compute:

$$d = a * b + c$$

That is not just arithmetic. It is also a graph:

- `a` and `b` flow into a multiply node
- the multiply result and `c` flow into an add node
- the final output is `d`

The framework's job is to remember enough of this structure so we can later send gradients backward through it.

In the next cell, build `a`, `b`, `c`, `e`, and `d` as `Value` objects.


## 3. Local Derivatives

Each operation only needs to know its **local rule**.

For addition:

$$\frac{\partial (x + y)}{\partial x} = 1, \quad \frac{\partial (x + y)}{\partial y} = 1$$

For multiplication:

$$\frac{\partial (x \cdot y)}{\partial x} = y, \quad \frac{\partial (x \cdot y)}{\partial y} = x$$

This is the core idea: each node stores a tiny backward rule for itself.

In code, that means each output node will carry a small closure that says how its parents should receive gradient.


## 4. Manual Backpropagation Intuition

Take a simple example:

$$e = a * b$$
$$d = e + c$$

If we want the gradient of `d` with respect to everything below it:

- `dd/dd = 1`
- `dd/de = 1` because `d = e + c`
- `dd/dc = 1`
- `dd/da = b`
- `dd/db = a`

So gradients flow backward one edge at a time.

In the next cell, seed `d.grad = 1.0`, then manually propagate gradients into `e`, `c`, `a`, and `b`.


## 5. What the Class Needs to Store

A useful first version of `Value` should keep track of:

- `data`: the actual number
- `grad`: the gradient flowing into that number
- `_prev`: the parent nodes used to create it
- `_op`: the operation that produced it

On Day 01, we can even do the backward pass manually for a tiny graph. On Day 02, we automate it with a topological traversal.

Use the next cell to flesh out the class shape and decide where `_prev`, `_op`, and `_backward` should live.


## 6. Operator Overloads to Implement

For Day 01, the API surface is intentionally tiny:

- `__init__`
- `__repr__`
- `__add__`
- `__mul__`

That is enough to make the graph real and the derivatives meaningful.

Use the next cell to write those methods from scratch.


## 7. Manual Gradient Check

A good teaching moment here is to compare:

- the derivatives you derive on paper
- the gradients produced by your manual `_backward` calls

Expected values for the toy graph:

- `dd/da = -3`
- `dd/db = 2`
- `dd/dc = 1`
- `dd/de = 1`


## 8. Prompt for the Video

Suggested flow while recording:

1. Start with plain numbers.
2. Show why gradients have nowhere to live.
3. Introduce the `Value` wrapper.
4. Implement `__add__` and `__mul__`.
5. Manually assign gradients on a toy graph.
6. Show a manual backward pass by calling `_backward` in reverse order.
7. Tease Day 02: automatic `.backward()`.


## 9. Homework

As a challenge, implement `__pow__` and derive its backward rule:

$$\frac{d(x^n)}{dx} = n x^{n-1}$$

The full prompt lives in `HOMEWORK.md`.
